# FPL historical research v2

The initial17-setting search did not prove optimal calibration. This notebook reads the completed, broader chronological research and provides its full reproduction command. The official API supplies season aggregates for current players; complete match panels come from the pinned archive. Research models are separate from the live advisor.


In [1]:
from pathlib import Path
import json
repo = Path.cwd()
if repo.name == "history": repo = repo.parent
base = repo / "history"
results = base / "results"
assert (results / "experiment.json").exists()
index = json.loads((base / "collections/latest.json").read_text())
print("Raw collection:", index["run_id"], "packages:", len(index["summary"]["packages"]))


Raw collection: 39909712054 packages: 11


## Official coverage and the limits of season aggregates

Rows below are totals for players still listed today, not complete old-season player populations or gameweek panels. The match-level archive contains ten complete seasons,2016/17–2025/26.


In [2]:
coverage = json.loads((results / "official-api-coverage.json").read_text())
print("Current players captured:", coverage["players_captured"])
print("Previous-season totals:", coverage["history_past_rows"])
for season, n in coverage["history_past_rows_by_season"].items(): print(season, n)


Current players captured: 668
Previous-season totals: 2110
2006/07 1
2007/08 3
2008/09 5
2009/10 5
2010/11 4
2011/12 7
2012/13 9
2013/14 14
2014/15 26
2015/16 29
2016/17 42
2017/18 61
2018/19 85
2019/20 109
2020/21 151
2021/22 174
2022/23 245
2023/24 293
2024/25 366
2025/26 481


## Chronological fitting checks

For each test year, all fitting years precede it. Nested validation also uses only earlier years. These checks do not make the retrospectively chosen model family a new untouched external holdout.


In [3]:
selection = json.loads((results / "nested-selection.json").read_text())
for target, audit in selection.items():
    assert all(year < target for year in audit["training_seasons"])
    if audit["inner_validation"]:
        assert audit["inner_validation"] < target
        assert all(year < audit["inner_validation"] for year in audit["inner_training_seasons"])
print("All training/validation temporal boundaries pass.")
current = selection["2026-27"]
print("Current model training rows:", current["training_rows"], "window:", current["selected"]["window"], "pseudocount:", current["selected"]["strength"])


All training/validation temporal boundaries pass.
Current model training rows: 228115 window: 10 pseudocount: 50


## Nine chronological minutes tests

Lower log loss is better. The first2017/18 cold-start fold fails badly; it is retained in the results. The position-constant diagnostic was added after initial results and did not select parameters.


In [4]:
minutes = json.loads((results / "minutes-walk-forward.json").read_text())
constant = json.loads((results / "position-constant-minute-benchmark.json").read_text())
for season, result in minutes.items():
    a = result["active"]
    print(season, "n=", a["learned"]["n"], "constant=", round(constant[season]["multiclass_logloss"], 4), "rolling=", round(a["rolling6"]["multiclass_logloss"], 4), "learned=", round(a["learned"]["multiclass_logloss"], 4))


2017-18 n= 7970 constant= 0.7293 rolling= 0.8016 learned= 1.0008
2018-19 n= 7959 constant= 0.7179 rolling= 0.8189 learned= 0.6733
2019-20 n= 8308 constant= 0.7267 rolling= 0.8445 learned= 0.648
2020-21 n= 7436 constant= 0.7315 rolling= 0.8466 learned= 0.6571
2021-22 n= 7096 constant= 0.7486 rolling= 0.8507 learned= 0.6678
2022-23 n= 7501 constant= 0.7524 rolling= 0.8138 learned= 0.6586
2023-24 n= 7876 constant= 0.7548 rolling= 0.8799 learned= 0.6573
2024-25 n= 8213 constant= 0.7625 rolling= 0.8272 learned= 0.6769
2025-26 n= 8133 constant= 0.7597 rolling= 0.88 learned= 0.6678


## Points forecasts and actual owned squads

The original v1 formula is retained; only the minute states change. Point recalibration uses earlier-year chronological predictions. Scoring-era differences and missing vintage odds/news remain limitations. Current seven-week decisions condition on the actual15 owned each week; they do not simulate a transfer policy.


In [5]:
points = json.loads((results / "points-walk-forward.json").read_text())
for season, models in points.items():
    print(season, "v1 RMSE:", round(models["v1"]["rmse"], 4), "learned:", round(models["learned_minutes"]["rmse"], 4), "calibrated:", round(models["learned_plus_points_calibration"]["rmse"], 4))
cohorts = json.loads((results / "current-cohorts.json").read_text())
for name, values in cohorts.items(): print(name, "current candidate RMSE:", round(values["fixed_candidate_pool"]["rmse"], 4), "owned RMSE:", round(values["owned15"]["rmse"], 4))
lineups = json.loads((results / "current-research-summary.json").read_text())
print("Research lineup totals:", {name:v["conditional_squad_score"] for name,v in lineups.items()})


2023-24 v1 RMSE: 3.0228 learned: 3.0011 calibrated: 3.0011
2024-25 v1 RMSE: 2.9206 learned: 2.9087 calibrated: 2.9109
2025-26 v1 RMSE: 3.0678 learned: 3.0234 calibrated: 3.0238
v1 current candidate RMSE: 3.1865 owned RMSE: 2.9723
learned_minutes current candidate RMSE: 3.1794 owned RMSE: 2.9845
learned_plus_points_calibration current candidate RMSE: 3.1771 owned RMSE: 2.9755
Research lineup totals: {'learned_minutes': 422.0, 'learned_plus_points_calibration': 422.0}


## Probability reliability and uncertainty

V1 underpredicts the average chance of reaching60 minutes on the fixed candidate pool. Learned probabilities improve but still underpredict. Bootstrap samples whole gameweeks; seven clusters and repeated research do not establish a deployment edge.


In [6]:
uncertainty = json.loads((results / "current-uncertainty.json").read_text())
for name, row in uncertainty["means"].items(): print(name, "predicted P60:", round(row["p60"], 4), "observed:", round(row["reached60"], 4))
for name, row in uncertainty["paired"].items():
    print(name, "P60 Brier improvement:", row["p60"], "points RMSE improvement:", row["xp"])


v1 predicted P60: 0.733 observed: 0.8224
learned_minutes predicted P60: 0.767 observed: 0.8224
learned_plus_points_calibration predicted P60: 0.767 observed: 0.8224
learned_minutes P60 Brier improvement: {'relative_error_reduction_percent': 12.286615093627951, 'gw_cluster_bootstrap_95_interval': [0.11865028373664321, 22.499694224810362], 'clusters': 7, 'n': 1205} points RMSE improvement: {'relative_error_reduction_percent': 0.22237621426909326, 'gw_cluster_bootstrap_95_interval': [-0.8592915473557738, 1.2484059719041891], 'clusters': 7, 'n': 1205}
learned_plus_points_calibration P60 Brier improvement: {'relative_error_reduction_percent': 12.286615093627951, 'gw_cluster_bootstrap_95_interval': [0.11865028373664321, 22.499694224810362], 'clusters': 7, 'n': 1205} points RMSE improvement: {'relative_error_reduction_percent': 0.29458153832699896, 'gw_cluster_bootstrap_95_interval': [-0.6972197615630726, 1.2229926240241762], 'clusters': 7, 'n': 1205}


## Explicit-season prior audit

Six frozen records conflict with both official2025/26 history and the archived season. None was in the seven owned squads. The proposal repairs source records for a future review; it does not replace old replay inputs.


In [7]:
audit = json.loads((results / "official-audit.json").read_text())
print("Outcome rows reconciled:", audit["current_outcome_rows_reconciled"], "differences:", len(audit["current_outcome_differences"]))
for row in audit["prior_differences"]: print(row["player_id"], row["name"], row["differences"]["mins"], "owned:", row["owned_in_current_seven"])
print("Live model unchanged:", json.loads((results / "reproduction-verification.json").read_text())["live_model_changed"] is False)


Outcome rows reconciled: 4551 differences: 0
54 Gomes {'frozen': '2207', 'official_history': 2829, 'archive_previous': '2829', 'archive_agrees': True} owned: False
105 Anthony {'frozen': '0', 'official_history': 2717, 'archive_previous': '2717', 'archive_agrees': True} owned: False
108 Wilson {'frozen': '0', 'official_history': 1234, 'archive_previous': '1234', 'archive_agrees': True} owned: False
153 Disasi {'frozen': '0', 'official_history': 1254, 'archive_previous': '1254', 'archive_agrees': True} owned: False
190 Tchaouna {'frozen': '0', 'official_history': 1259, 'archive_previous': '1259', 'archive_agrees': True} owned: False
571 Florentino {'frozen': '0', 'official_history': 2100, 'archive_previous': '2100', 'archive_agrees': True} owned: False
Live model unchanged: True


## Full reproduction

Run from the repository root with Python3.12 standard library only. The script verifies the committed raw packages, downloads pinned current replay inputs, fits the chronological models and writes ignored work outputs. Enable the flag to execute the full run.


In [8]:
RUN_FULL = False
if RUN_FULL:
    import subprocess
    subprocess.run(["bash", "history/run_research.sh"], cwd=repo, check=True)
else:
    print("Reproduction command: bash history/run_research.sh")


Reproduction command: bash history/run_research.sh
